# Day 1 - The Core Structures & Loading Data

## Series vs DataFrame

A `Series` is a single labelled column. The numbers on the left (0,1,2,3) are the index, the label for each row. This is the key difference from NumPy where rows are only identified by position. In Pandas every row has a label.

In [73]:
import pandas as pd
import numpy as np

s = pd.Series([3.7, 3.6, 3.8, 3.5], name='voltage')
print(s)

0    3.7
1    3.6
2    3.8
3    3.5
Name: voltage, dtype: float64


Pandas the index is attached to the data like a sticky label, it travels with the row wherever it goes. So if you sort or filter you can absolutely end up with:

In [74]:
# df.sort_values('voltage')
#    voltage
# 3    3.5     ← row 3 is now first
# 1    3.6     ← row 1 is second
# 0    3.7     ← row 0 is third
# 2    3.8     ← row 2 is last

A `DataFrame` is a full table of multiple Series side by side. <br>Creating DataFrames from a dictionary is shown below:


In [75]:
# From a dictionary — most common
df = pd.DataFrame({
    'voltage':     [3.7, 3.6, 3.8, 3.5],
    'current':     [10.2, 10.1, 10.5, 9.8],
    'temperature': [24.5, 24.8, 25.1, 24.9]
})

# From a NumPy array — you'll use this when converting existing data
data = np.array([[3.7, 10.2, 24.5],
                 [3.6, 10.1, 24.8],
                 [3.8, 10.5, 25.1]])

df = pd.DataFrame(data, columns=['voltage', 'current', 'temperature'])

# From a CSV — the most common in real projects
# df = pd.read_csv('data.csv')

**The 5 Exploration Commands**
<br> Run these on every dataset before touching anything.

df.info() is the most important one. It tells you:

- What columns exist
- What data type each column is (int, float, object, datetime)
- How many non-null values — which means you can immediately see where data is missing

In [76]:
df.head()       # first 5 rows
df.tail()       # last 5 rows
df.info()       # column names, data types, how many non-null values
df.describe()   # min, max, mean, std for every numerical column
df.shape        # (rows, columns) — note: no brackets, it's a property not a function

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3 entries, 0 to 2
Data columns (total 3 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   voltage      3 non-null      float64
 1   current      3 non-null      float64
 2   temperature  3 non-null      float64
dtypes: float64(3)
memory usage: 204.0 bytes


(3, 3)

## Selecting Data


Single column — returns a Series:

In [77]:
df['voltage']

0    3.7
1    3.6
2    3.8
Name: voltage, dtype: float64

Multiple columns — returns a DataFrame:

In [78]:
df[['voltage', 'current']]

,voltage,current
0,3.7,10.2
1,3.6,10.1
2,3.8,10.5


`iloc` vs `loc` <br>
Two ways to select rows — this is the one that confuses people most at first. <br> <br>
`iloc` — select by integer position, exactly like NumPy:

In [79]:
df.iloc[0]          # first row
df.iloc[-1]         # last row
df.iloc[0:3]        # rows 0, 1, 2
df.iloc[0, 1]       # row 0, column 1 — single value
df.iloc[0:3, 0:2]   # rows 0-2, columns 0-1

,voltage,current
0,3.7,10.2
1,3.6,10.1
2,3.8,10.5


`loc` — select by label:


In [80]:
df.loc[0]                    # row with index label 0
df.loc[0, 'voltage']         # row 0, voltage column
df.loc[0:3, 'voltage']       # rows 0 to 3, voltage column
df.loc[0:3, ['voltage', 'current']]   # rows 0-3, two columns

,voltage,current
0,3.7,10.2
1,3.6,10.1
2,3.8,10.5


**Simple rule: use `iloc` when you want position, use `loc` when you want a specific label.**


## Boolean Masking — Same as NumPy


In [81]:
# Single condition
df[df['voltage'] > 3.6]
#    voltage  current  temperature
# 0      3.7     10.2         24.5
# 2      3.8     10.5         25.1

# Multiple conditions — same & and | rules as NumPy
df[(df['voltage'] > 3.6) & (df['temperature'] < 25.0)]

# Negation — rows where condition is False
df[~(df['voltage'] > 3.6)]    # ~ means NOT

,voltage,current,temperature
1,3.6,10.1,24.8


The only real difference

**NumPy — you refer to columns by position number** <br> <br>
    `data[:, 0]`     # column 0 = voltage <br>
    `data[:, 2]`    # column 2 = temperature

**Pandas — you refer to columns by name** <br> <br>
    `df['voltage']` <br>
    `df['temperature']`

## Adding and Modifying Columns


In [82]:
print("BEFORE:")
print(df)

BEFORE:
   voltage  current  temperature
0      3.7     10.2         24.5
1      3.6     10.1         24.8
2      3.8     10.5         25.1


In [83]:
# Add a new column
df['power_w'] = df['voltage'] * df['current']

# Modify an existing column
df['voltage'] = df['voltage'].round(2)

# Add a constant column
df['team'] = 'High Speed Karlsruhe'


print("AFTER:")

print(df)

AFTER:
   voltage  current  temperature  power_w                  team
0      3.7     10.2         24.5    37.74  High Speed Karlsruhe
1      3.6     10.1         24.8    36.36  High Speed Karlsruhe
2      3.8     10.5         25.1    39.90  High Speed Karlsruhe


## Dropping Columns and Rows


In [84]:
df = pd.DataFrame({
    'voltage':     [3.7, 3.6, 3.8, 3.5],
    'current':     [10.2, 10.1, 10.5, 9.8],
    'temperature': [24.5, 24.8, 25.1, 24.9],
    'team':        ['HSK', 'HSK', 'HSK', 'HSK']
})

print("BEFORE:")
print(df)

BEFORE:
   voltage  current  temperature team
0      3.7     10.2         24.5  HSK
1      3.6     10.1         24.8  HSK
2      3.8     10.5         25.1  HSK
3      3.5      9.8         24.9  HSK


**1) Drop a column**

In [85]:
df.drop(columns=['team'], inplace=True)
print(df)

   voltage  current  temperature
0      3.7     10.2         24.5
1      3.6     10.1         24.8
2      3.8     10.5         25.1
3      3.5      9.8         24.9


**2) Drop a row**

In [86]:
df.drop(index=2, inplace=True)
print(df)

   voltage  current  temperature
0      3.7     10.2         24.5
1      3.6     10.1         24.8
3      3.5      9.8         24.9


**3) — Drop multiple rows**

In [87]:
df.drop(index=[1, 3], inplace=True)
print(df)

   voltage  current  temperature
0      3.7     10.2         24.5


## Renaming Columns

In [88]:
df.rename(columns={'voltage': 'V', 
                   'current': 'I', 
                   'temperature': 'T'}, inplace=True)
print(df)

     V     I     T
0  3.7  10.2  24.5


## Checking Value Counts
Very useful for categorical columns — tells you how many times each unique value appears:

In [89]:
df = pd.DataFrame({
    'driver':      ['Soman', 'Ahmed', 'Lars', 'Soman', 'Ahmed', 
                    'Soman', 'Lars', 'Ahmed', 'Soman', 'Lars'],
    'lap':         [1, 1, 1, 2, 2, 2, 3, 3, 3, 3],
    'voltage':     [3.71, 3.65, 3.78, 3.69, 3.72, 3.61, 3.74, 3.68, 3.70, 3.63],
    'speed_kmh':   [82.0, 78.5, 91.2, 85.1, 79.3, 88.7, 83.4, 77.9, 86.2, 90.1]
})

print(df)

  driver  lap  voltage  speed_kmh
0  Soman    1     3.71       82.0
1  Ahmed    1     3.65       78.5
2   Lars    1     3.78       91.2
3  Soman    2     3.69       85.1
4  Ahmed    2     3.72       79.3
5  Soman    2     3.61       88.7
6   Lars    3     3.74       83.4
7  Ahmed    3     3.68       77.9
8  Soman    3     3.70       86.2
9   Lars    3     3.63       90.1


In [90]:
df['driver'].value_counts()


driver
Soman    4
Ahmed    3
Lars     3
Name: count, dtype: int64

In [91]:
df['driver'].unique() 
df['driver'].nunique()

3

## PRACTICE EXERCISE

In [92]:
import pandas as pd
import numpy as np

np.random.seed(42)

df = pd.DataFrame({
    'driver':      ['Soman', 'Ahmed', 'Lars', 'Soman', 'Ahmed', 'Lars'],
    'lap':         [1, 1, 1, 2, 2, 2],
    'voltage':     [3.71, 3.65, 3.78, 3.69, 3.72, 3.61],
    'current':     [10.2, 9.8, 11.1, 10.5, 9.9, 10.8],
    'temperature': [24.5, 25.1, 23.8, 24.9, 25.3, 24.1],
    'speed_kmh':   [82.0, 78.5, 91.2, 85.1, 79.3, 88.7]
})
print(df)

  driver  lap  voltage  current  temperature  speed_kmh
0  Soman    1     3.71     10.2         24.5       82.0
1  Ahmed    1     3.65      9.8         25.1       78.5
2   Lars    1     3.78     11.1         23.8       91.2
3  Soman    2     3.69     10.5         24.9       85.1
4  Ahmed    2     3.72      9.9         25.3       79.3
5   Lars    2     3.61     10.8         24.1       88.7


1. Select only the voltage and current columns
2. Find all rows where speed is above 85
3. Find all rows where Soman is driving AND temperature is below 25
4. Add a new column called power_w that is voltage multiplied by current
5. Find the row with the highest speed using idxmax() — figure out what that does

1. Select only the voltage and current columns

In [93]:
df[['voltage', 'current']]

,voltage,current
0,3.71,10.2
1,3.65,9.8
2,3.78,11.1
3,3.69,10.5
4,3.72,9.9
5,3.61,10.8


2. Find all rows where speed is above 85

In [94]:
df[df['speed_kmh'] > 85.0]


,driver,lap,voltage,current,temperature,speed_kmh
2,Lars,1,3.78,11.1,23.8,91.2
3,Soman,2,3.69,10.5,24.9,85.1
5,Lars,2,3.61,10.8,24.1,88.7


3. Find all rows where Soman is driving AND temperature is below 25

In [95]:
df[(df['driver'] == 'Soman') & (df['temperature'] < 25.0)]

,driver,lap,voltage,current,temperature,speed_kmh
0,Soman,1,3.71,10.2,24.5,82.0
3,Soman,2,3.69,10.5,24.9,85.1


4. Add a new column called power_w that is voltage multiplied by current

In [96]:
df['power_w'] = df['voltage'] * df['current']
print(df)

  driver  lap  voltage  current  temperature  speed_kmh  power_w
0  Soman    1     3.71     10.2         24.5       82.0   37.842
1  Ahmed    1     3.65      9.8         25.1       78.5   35.770
2   Lars    1     3.78     11.1         23.8       91.2   41.958
3  Soman    2     3.69     10.5         24.9       85.1   38.745
4  Ahmed    2     3.72      9.9         25.3       79.3   36.828
5   Lars    2     3.61     10.8         24.1       88.7   38.988


5. Find the row with the highest speed using idxmax() and figure out what that does

In [97]:
highest_speed = df['speed_kmh'].idxmax()
print(f"Highest speed is {df.loc[highest_speed, 'speed_kmh']}")

Highest speed is 91.2


# Day 2 - Missing Data

This is one of the most important practical skills in all of ML. Real world data always has gaps; a sensor dropout, a corrupted reading, an empty spreadsheet cell. Before you can do any analysis or feed data into a model you need to handle these. First let's create a DataFrame with realistic missing values so you can see everything in action:

In [98]:
import pandas as pd
import numpy as np

df = pd.DataFrame({
    'driver':      ['Soman', 'Ahmed', 'Lars', 'Soman', 'Ahmed', 'Lars'],
    'lap':         [1, 1, 1, 2, 2, 2],
    'voltage':     [3.71, np.nan, 3.78, 3.69, 3.72, 3.61],
    'current':     [10.2, 10.1, np.nan, 10.5, np.nan, 10.8],
    'temperature': [24.5, 25.1, 23.8, np.nan, 25.3, 24.1],
    'speed_kmh':   [82.0, 78.5, 91.2, 85.1, 79.3, np.nan]
})

print(df)

  driver  lap  voltage  current  temperature  speed_kmh
0  Soman    1     3.71     10.2         24.5       82.0
1  Ahmed    1      NaN     10.1         25.1       78.5
2   Lars    1     3.78      NaN         23.8       91.2
3  Soman    2     3.69     10.5          NaN       85.1
4  Ahmed    2     3.72      NaN         25.3       79.3
5   Lars    2     3.61     10.8         24.1        NaN


`np.nan` is how you represent a missing value — it stands for "Not a Number". Notice it displays as NaN in the DataFrame.

## Step 1 — Find where the gaps are


In [99]:
# True/False for every single cell
print(df.isnull())

   driver    lap  voltage  current  temperature  speed_kmh
0   False  False    False    False        False      False
1   False  False     True    False        False      False
2   False  False    False     True        False      False
3   False  False    False    False         True      False
4   False  False    False     True        False      False
5   False  False    False    False        False       True


Easy to read version below using count

In [100]:
# Count of missing values per column
print(df.isnull().sum())

driver         0
lap            0
voltage        1
current        2
temperature    1
speed_kmh      1
dtype: int64


In [101]:
# Total missing in entire dataset
print(df.isnull().sum().sum())

5


In [102]:
# What percentage of each column is missing
print((df.isnull().sum() / len(df) * 100).round(1))

driver          0.0
lap             0.0
voltage        16.7
current        33.3
temperature    16.7
speed_kmh      16.7
dtype: float64



This last one is the most useful in practice, it tells you how severe the missing data problem is per column. If a column is 80% missing, you might just drop it entirely. If it's 5% missing, filling is fine.

## Step 2 — Three strategies for handling missing values

### Strategy 1 — Drop rows:


In [103]:
df_dropped = df.dropna() #Drops all rows that include missing values
print(df_dropped)

  driver  lap  voltage  current  temperature  speed_kmh
0  Soman    1     3.71     10.2         24.5       82.0


In [104]:
# Only drop if voltage specifically is missing
df.dropna(subset=['voltage'])

,driver,lap,voltage,current,temperature,speed_kmh
0,Soman,1,3.71,10.2,24.5,82.0
2,Lars,1,3.78,NaN,23.8,91.2
3,Soman,2,3.69,10.5,NaN,85.1
4,Ahmed,2,3.72,NaN,25.3,79.3
5,Lars,2,3.61,10.8,24.1,NaN


In [105]:
# Only drop if ALL values in the row are missing
df.dropna(how='all')

,driver,lap,voltage,current,temperature,speed_kmh
0,Soman,1,3.71,10.2,24.5,82.0
1,Ahmed,1,NaN,10.1,25.1,78.5
2,Lars,1,3.78,NaN,23.8,91.2
3,Soman,2,3.69,10.5,NaN,85.1
4,Ahmed,2,3.72,NaN,25.3,79.3
5,Lars,2,3.61,10.8,24.1,NaN


### Strategy 2 — Fill with a statistical value:


Use mean/median filling when missing values are random, a sensor glitch that's unrelated to the actual conditions.

In [106]:
# Fill one column with its mean
df['voltage'] = df['voltage'].fillna(df['voltage'].mean())
print(df)

  driver  lap  voltage  current  temperature  speed_kmh
0  Soman    1    3.710     10.2         24.5       82.0
1  Ahmed    1    3.702     10.1         25.1       78.5
2   Lars    1    3.780      NaN         23.8       91.2
3  Soman    2    3.690     10.5          NaN       85.1
4  Ahmed    2    3.720      NaN         25.3       79.3
5   Lars    2    3.610     10.8         24.1        NaN


In [107]:
# Fill one column with median — better when data has outliers
df['current'] = df['current'].fillna(df['current'].median())
print(df)

  driver  lap  voltage  current  temperature  speed_kmh
0  Soman    1    3.710    10.20         24.5       82.0
1  Ahmed    1    3.702    10.10         25.1       78.5
2   Lars    1    3.780    10.35         23.8       91.2
3  Soman    2    3.690    10.50          NaN       85.1
4  Ahmed    2    3.720    10.35         25.3       79.3
5   Lars    2    3.610    10.80         24.1        NaN


In [108]:
# Fill all numerical columns at once with their means
df.fillna(df.mean(numeric_only=True), inplace=True)
print(df)

  driver  lap  voltage  current  temperature  speed_kmh
0  Soman    1    3.710    10.20        24.50      82.00
1  Ahmed    1    3.702    10.10        25.10      78.50
2   Lars    1    3.780    10.35        23.80      91.20
3  Soman    2    3.690    10.50        24.56      85.10
4  Ahmed    2    3.720    10.35        25.30      79.30
5   Lars    2    3.610    10.80        24.10      83.22


### Strategy 3 — Forward fill `ffill`:


This replaces a missing value with the last known value above it. For time-series sensor data this is the most physically realistic; if a speed reading drops out for one sample, the car was probably still going roughly the same speed.

In [109]:
df['speed_kmh'] = df['speed_kmh'].fillna(method='ffill')
print(df)

  driver  lap  voltage  current  temperature  speed_kmh
0  Soman    1    3.710    10.20        24.50      82.00
1  Ahmed    1    3.702    10.10        25.10      78.50
2   Lars    1    3.780    10.35        23.80      91.20
3  Soman    2    3.690    10.50        24.56      85.10
4  Ahmed    2    3.720    10.35        25.30      79.30
5   Lars    2    3.610    10.80        24.10      83.22


/var/folders/7b/77f4srw15n788bj4vzg5sm3w0000gn/T/ipykernel_55207/722007470.py:1: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df['speed_kmh'] = df['speed_kmh'].fillna(method='ffill')


There's also `bfill` — backward fill, uses the next value below instead. Less common for sensor data.

## Most common mistake

In [110]:
# This does NOTHING to df — result is thrown away
df['voltage'].fillna(df['voltage'].mean())

# You must either reassign
df['voltage'] = df['voltage'].fillna(df['voltage'].mean())

# Or use inplace=True
df['voltage'].fillna(df['voltage'].mean(), inplace=True)

/var/folders/7b/77f4srw15n788bj4vzg5sm3w0000gn/T/ipykernel_55207/132927668.py:8: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['voltage'].fillna(df['voltage'].mean(), inplace=True)


## PRACTICE EXERCISE

In [111]:
df = pd.DataFrame({
    'driver':      ['Soman', 'Ahmed', 'Lars', 'Soman', 'Ahmed', 'Lars'],
    'lap':         [1, 1, 1, 2, 2, 2],
    'voltage':     [3.71, np.nan, 3.78, 3.69, 3.72, 3.61],
    'current':     [10.2, 10.1, np.nan, 10.5, np.nan, 10.8],
    'temperature': [24.5, 25.1, 23.8, np.nan, 25.3, 24.1],
    'speed_kmh':   [82.0, 78.5, 91.2, 85.1, 79.3, np.nan]
})

1. Find how many missing values are in each column
2. Find what percentage of current is missing
3. Fill voltage with its mean
4. Fill speed_kmh using forward fill
5. Fill temperature with its median
6. Confirm there are no missing values left with isnull().sum()

1. Find how many missing values are in each column

In [112]:
print(df.isnull().sum())

driver         0
lap            0
voltage        1
current        2
temperature    1
speed_kmh      1
dtype: int64


2. Find what percentage of current is missing


In [113]:
print((df.isnull().sum() / len(df) * 100).round(1))

driver          0.0
lap             0.0
voltage        16.7
current        33.3
temperature    16.7
speed_kmh      16.7
dtype: float64


3. Fill voltage with its mean


In [114]:
df['voltage'] = df['voltage'].fillna(df['voltage'].mean())
print(df)

  driver  lap  voltage  current  temperature  speed_kmh
0  Soman    1    3.710     10.2         24.5       82.0
1  Ahmed    1    3.702     10.1         25.1       78.5
2   Lars    1    3.780      NaN         23.8       91.2
3  Soman    2    3.690     10.5          NaN       85.1
4  Ahmed    2    3.720      NaN         25.3       79.3
5   Lars    2    3.610     10.8         24.1        NaN


4. Fill speed_kmh using forward fill

In [115]:
df['speed_kmh'] = df['speed_kmh'].fillna(method='ffill')
print(df)

  driver  lap  voltage  current  temperature  speed_kmh
0  Soman    1    3.710     10.2         24.5       82.0
1  Ahmed    1    3.702     10.1         25.1       78.5
2   Lars    1    3.780      NaN         23.8       91.2
3  Soman    2    3.690     10.5          NaN       85.1
4  Ahmed    2    3.720      NaN         25.3       79.3
5   Lars    2    3.610     10.8         24.1       79.3


/var/folders/7b/77f4srw15n788bj4vzg5sm3w0000gn/T/ipykernel_55207/722007470.py:1: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df['speed_kmh'] = df['speed_kmh'].fillna(method='ffill')


5. Fill temperature with its median


In [116]:
df['temperature'] = df['temperature'].fillna(df['temperature'].median())
print(df)

  driver  lap  voltage  current  temperature  speed_kmh
0  Soman    1    3.710     10.2         24.5       82.0
1  Ahmed    1    3.702     10.1         25.1       78.5
2   Lars    1    3.780      NaN         23.8       91.2
3  Soman    2    3.690     10.5         24.5       85.1
4  Ahmed    2    3.720      NaN         25.3       79.3
5   Lars    2    3.610     10.8         24.1       79.3


6. Confirm there are no missing values left with isnull().sum()

In [117]:
print(df.isnull().sum())

driver         0
lap            0
voltage        0
current        2
temperature    0
speed_kmh      0
dtype: int64


In [118]:
df['current'] = df['current'].fillna(df['current'].mean())
print(df.isnull().sum())

driver         0
lap            0
voltage        0
current        0
temperature    0
speed_kmh      0
dtype: int64
